# EXP-008 (hướng A) — Tiền xử lý ảnh lúc test + TTA lật ngang
Không train. Đổi khâu inference, giữ nguyên checkpoint:
- **P0**: như `test.py` hiện tại — `Resize((224,224))` bilinear + `ToTensor` (**không** chuẩn hoá CLIP)
- **P1**: P0 + `Normalize` mean/std của CLIP
- **P2**: `Resize((224,224))` **bicubic** + `ToTensor` + `Normalize` (giống preprocess của `clip.load` với ảnh vuông)
- **Flip**: trung bình đặc trưng ảnh CLIP của ảnh gốc và ảnh lật ngang (trước head)

→ 6 cấu hình × head: CLIP gốc · Paper · EXP-000 (3 seed) · EXP-004 λ=1 ep30 (3 seed). Prompt chính Expression Ensemble Five, phụ Ensemble Ten.

**Tiêu chí (chọn trước):** cấu hình chính **P0 + Flip**: Mean4 UAR ≥ +0.5 so với P0 trên cả EXP-000 và EXP-004, không dataset nào < −0.5.
Chuẩn hoá (P1/P2) là chẩn đoán: CLIP gốc tăng ≥ +1.0 Mean4 → có cơ sở train lại với tiền xử lý đúng (hướng B).

# Cấu hình

In [1]:
SEEDS = [1, 2, 3]
BRANCH = "exp/008-test-preprocessing-tta"   # chỉ cần models/clip + models/Text.py (không cần lavis)
TEST_ZIP = "/content/drive/MyDrive/Datasets/static_test_sets.zip"
OUT_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp008"
EXP000_ROOT = "/content/drive/MyDrive/ExpCLIP_output"  # tìm đệ quy expclip_l14_face_seed{N}-model.pth
EXP004_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp004"
DATASETS = ["RAFDB", "AffectNet7", "AffectNet8", "FERPlus"]
N_CLS = {"RAFDB": 7, "AffectNet7": 7, "AffectNet8": 8, "FERPlus": 8}
PROMPT_NAMES = ["Expression Ensemble Five", "Expression Ensemble Ten"]
BASE_PROMPT = "Expression Ensemble Five"
PREPS = ["P0", "P1", "P2"]
CONFIGS = [(p, f) for p in PREPS for f in (False, True)]   # (tiền xử lý, flip)
BASE_CFG, MAIN_CFG = "P0", "P0+flip"

# Mount Drive & GPU

In [2]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA L4, 23034 MiB


# Clone repo & giải nén test set

In [3]:
if not os.path.isdir("/content/Exp-CLIP"):
    !git clone -b $BRANCH https://github.com/huongng389/Exp-CLIP.git /content/Exp-CLIP
!cd /content/Exp-CLIP && git checkout $BRANCH && git pull && git log --oneline -1   # ghi lại commit vào research_log
!mkdir -p /content/static_test_sets
!unzip -q -n "$TEST_ZIP" -d /content/static_test_sets
!ls /content/static_test_sets/*/test
# CLIP vendored cần ftfy và pkg_resources (setuptools mới đã bỏ)
!pip install -q ftfy "setuptools<81"

Cloning into '/content/Exp-CLIP'...
remote: Enumerating objects: 441, done.
remote: Counting objects: 100% (441/441), done.
remote: Compressing objects: 100% (167/167), done.
remote: Total 441 (delta 323), reused 387 (delta 270), pack-reused 0 (from 0)
Receiving objects: 100% (441/441), 32.29 MiB | 11.08 MiB/s, done.
Resolving deltas: 100% (323/323), done.
Already on 'exp/008-test-preprocessing-tta'
Your branch is up to date with 'origin/exp/008-test-preprocessing-tta'.
Already up to date.
2c95de1 (HEAD -> exp/008-test-preprocessing-tta, origin/exp/008-test-preprocessing-tta, origin/exp/007-au-instruction-prompts) EXP-007: add executed AU-prompt test notebook (branch A results)
/content/static_test_sets/AffectNet7/test:
0_Happiness  1_Sadness	2_Neutral  3_Anger  4_Surprise	5_Disgust  6_Fear

/content/static_test_sets/AffectNet8/test:
0_Happiness  2_Neutral	4_Surprise  6_Fear
1_Sadness    3_Anger	5_Disgust   7_Contempt

/content/static_test_sets/FERPlus/test:
0_Happiness  2_Neutral	4_Su

# Tìm checkpoint
Thiếu file nào thì bỏ qua (in cảnh báo).

In [4]:
import glob

def find_ckpt(root, name):
    hits = glob.glob(f"{root}/**/{name}", recursive=True)
    return hits[0] if hits else None

HEADS = {
    "CLIP gốc": {0: None},
    "Paper": {0: "/content/Exp-CLIP/checkpoint/ExpCLIP_L14_model.pth"},
    "EXP-000": {s: find_ckpt(EXP000_ROOT, f"expclip_l14_face_seed{s}-model.pth") for s in SEEDS},
    "EXP-004": {s: f"{EXP004_DIR}/exp004_anchor_crossA_l1_l14_seed{s}-e30-model.pth" for s in SEEDS},
}
for m, d in HEADS.items():
    for s, p in list(d.items()):
        if p is not None and not os.path.exists(p):
            print(f"CẢNH BÁO: {m} seed {s}: không thấy checkpoint ({p}) → bỏ qua")
            del d[s]
        else:
            print(f"{m} seed {s}: {p}")

CLIP gốc seed 0: None
Paper seed 0: /content/Exp-CLIP/checkpoint/ExpCLIP_L14_model.pth
EXP-000 seed 1: /content/drive/MyDrive/ExpCLIP_output/expclip_l14_face_seed1-model.pth
EXP-000 seed 2: /content/drive/MyDrive/ExpCLIP_output/expclip_l14_face_seed2-model.pth
EXP-000 seed 3: /content/drive/MyDrive/ExpCLIP_output/expclip_l14_face_seed3-model.pth
EXP-004 seed 1: /content/drive/MyDrive/ExpCLIP_output/exp004/exp004_anchor_crossA_l1_l14_seed1-e30-model.pth
EXP-004 seed 2: /content/drive/MyDrive/ExpCLIP_output/exp004/exp004_anchor_crossA_l1_l14_seed2-e30-model.pth
EXP-004 seed 3: /content/drive/MyDrive/ExpCLIP_output/exp004/exp004_anchor_crossA_l1_l14_seed3-e30-model.pth


# Feature CLIP L/14
Ảnh: mỗi tiền xử lý × (gốc, lật) tính 1 lần. Text: 2 prompt × 4 dataset (không phụ thuộc tiền xử lý ảnh).

In [5]:
import sys, json, numpy as np, pandas as pd, torch, torch.nn.functional as F
import torchvision.datasets as datasets, torchvision.transforms as T
sys.path.insert(0, "/content/Exp-CLIP")
from models.clip import clip
from models.Text import FER_prompt_list, FER_prompt_type_list

PROMPTS = {pn: pl for pn, pl in zip(FER_prompt_type_list, FER_prompt_list) if pn in PROMPT_NAMES}
assert list(PROMPTS) == PROMPT_NAMES, PROMPTS.keys()
clip_model, _ = clip.load("ViT-L/14", "cuda")
clip_model.eval()

CLIP_MEAN, CLIP_STD = (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)
BICUBIC = T.InterpolationMode.BICUBIC
TRANSFORMS = {
    "P0": T.Compose([T.Resize((224, 224)), T.ToTensor()]),                                   # giống test.py
    "P1": T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(CLIP_MEAN, CLIP_STD)]),
    "P2": T.Compose([T.Resize((224, 224), interpolation=BICUBIC), T.ToTensor(), T.Normalize(CLIP_MEAN, CLIP_STD)]),
}

img, y, classes, txt = {}, {}, {}, {}   # img[(ds, prep)] = (feat gốc, feat lật)
with torch.no_grad():
    for ds_name in DATASETS:
        for prep, tf in TRANSFORMS.items():
            ds = datasets.ImageFolder(f"/content/static_test_sets/{ds_name}/test", tf)
            loader = torch.utils.data.DataLoader(ds, batch_size=256, shuffle=False, num_workers=4)
            f0, f1 = [], []
            for x, _ in loader:
                x = x.cuda()
                f0.append(clip_model.encode_image(x).float().cpu())
                f1.append(clip_model.encode_image(x.flip(-1)).float().cpu())   # lật ngang
            img[(ds_name, prep)] = (torch.cat(f0), torch.cat(f1))
        y[ds_name] = np.array(ds.targets)
        classes[ds_name] = [c.split("_", 1)[1] for c in ds.classes]   # 0_Happiness → Happiness
        for pn, pl in PROMPTS.items():
            txt[(ds_name, pn)] = clip_model.encode_text(
                clip.tokenize(pl[ds_name], context_length=77, truncate=True).cuda()).float().cpu()
        print(ds_name, tuple(img[(ds_name, "P0")][0].shape), classes[ds_name])

/content/Exp-CLIP/models/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
100%|███████████████████████████████████████| 890M/890M [00:15<00:00, 61.8MiB/s]


RAFDB (3068, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear']
AffectNet7 (3500, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear']
AffectNet8 (4000, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear', 'Contempt']
FERPlus (3573, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear', 'Contempt']


# Dự đoán & metric

In [6]:
def load_W(path):
    if path is None:
        return None
    sd = torch.load(path, map_location="cpu")
    return next(v for k, v in sd.items() if "mlp.weight" in k).float()   # (4096, 768)

def cfg_name(prep, flip):
    return f"{prep}+flip" if flip else prep

def predict(W, ds_name, prep, flip, pn):
    f0, f1 = img[(ds_name, prep)]
    i, t = ((f0 + f1) / 2 if flip else f0), txt[(ds_name, pn)]   # TTA: trung bình đặc trưng CLIP trước head
    if W is not None:
        i, t = i @ W.T, t @ W.T
    i, t = F.normalize(i, dim=-1), F.normalize(t, dim=-1)
    return (i @ t.T).view(len(i), N_CLS[ds_name], -1).mean(-1).argmax(1).numpy()   # prompt xếp theo lớp, như test.py

def metrics(y_, p_, n_cls):
    acc = np.array([(p_[y_ == c] == c).mean() * 100 for c in range(n_cls)])
    return {"UAR": acc.mean(), "WAR": (p_ == y_).mean() * 100, "acc": acc}

rows = []
for m, d in HEADS.items():
    for s, path in d.items():
        W = load_W(path)
        for ds_name in DATASETS:
            for prep, flip in CONFIGS:
                for pn in PROMPT_NAMES:
                    r = metrics(y[ds_name], predict(W, ds_name, prep, flip, pn), N_CLS[ds_name])
                    rows.append({"model": m, "seed": s, "dataset": ds_name, "config": cfg_name(prep, flip), "prompt": pn,
                                 "UAR": r["UAR"], "WAR": r["WAR"],
                                 **{f"acc_{c}": a for c, a in zip(classes[ds_name], r["acc"])}})
res = pd.DataFrame(rows)
res.to_csv(f"{OUT_DIR}/exp008_test_preprocessing-results.csv", index=False)
CFG_NAMES = [cfg_name(p, f) for p, f in CONFIGS]
print(len(res), "dòng →", OUT_DIR)

384 dòng → /content/drive/MyDrive/ExpCLIP_output/exp008


# Kiểm tra: P0 không flip + Ensemble Five phải khớp `test.py`
So với kết quả đã ghi của EXP-000 / EXP-004. Lệch ≤ ~0.4 là do GPU fp16 (EXP-007 lệch tối đa 0.39); lệch lớn thì pipeline sai, dừng lại.

In [ ]:
REF = {"EXP-000|1|RAFDB": [58.88, 66.88], "EXP-000|1|AffectNet7": [44.09, 44.09], "EXP-000|1|AffectNet8": [39.23, 39.23], "EXP-000|1|FERPlus": [40.65, 51.47], "EXP-000|2|RAFDB": [57.82, 65.78], "EXP-000|2|AffectNet7": [43.69, 43.69], "EXP-000|2|AffectNet8": [38.17, 38.17], "EXP-000|2|FERPlus": [41.08, 53.6], "EXP-000|3|RAFDB": [57.5, 63.3], "EXP-000|3|AffectNet7": [44.74, 44.74], "EXP-000|3|AffectNet8": [39.12, 39.12], "EXP-000|3|FERPlus": [42.93, 55.16], "EXP-004|1|RAFDB": [58.73, 65.94], "EXP-004|1|AffectNet7": [45.83, 45.83], "EXP-004|1|AffectNet8": [39.6, 39.6], "EXP-004|1|FERPlus": [46.14, 52.03], "EXP-004|2|RAFDB": [57.07, 64.77], "EXP-004|2|AffectNet7": [43.97, 43.97], "EXP-004|2|AffectNet8": [38.75, 38.75], "EXP-004|2|FERPlus": [45.63, 52.45], "EXP-004|3|RAFDB": [58.26, 65.06], "EXP-004|3|AffectNet7": [45.46, 45.46], "EXP-004|3|AffectNet8": [39.07, 39.08], "EXP-004|3|FERPlus": [45.34, 51.44]}

chk = []
for k, (u, w) in REF.items():
    m, s, ds_name = k.split("|")
    r = res[(res.model == m) & (res.seed == int(s)) & (res.dataset == ds_name) & (res.config == BASE_CFG) & (res.prompt == BASE_PROMPT)]
    if len(r):
        chk.append({"key": k, "dUAR": r.UAR.item() - u, "dWAR": r.WAR.item() - w})
chk = pd.DataFrame(chk)
print(f"{len(chk)}/{len(REF)} cấu hình so được | max |ΔUAR| = {chk.dUAR.abs().max():.2f}, max |ΔWAR| = {chk.dWAR.abs().max():.2f}")
print(chk.round(2).sort_values("dUAR", key=abs, ascending=False).head(5).to_string(index=False))

# Bảng 1 — UAR theo cấu hình × dataset (Ensemble Five, mean ± std qua seed)

In [8]:
pd.set_option("display.width", 250)
def fmt(x):
    return f"{x.mean():.2f}" if len(x) == 1 else f"{x.mean():.2f} ± {x.std(ddof=1):.2f}"

u = res.pivot_table(index=["model", "seed", "prompt", "config"], columns="dataset", values="UAR")[DATASETS]
u["Mean4"] = u.mean(1)

def table(pn, delta=False):
    for m in HEADS:
        x = u.loc[m].xs(pn, level="prompt")
        if delta:
            x = x - x.xs(BASE_CFG, level="config").reindex(x.index, level="seed")
        t = x.groupby(level="config").agg(fmt).loc[[c for c in CFG_NAMES if not (delta and c == BASE_CFG)]]
        print(f"--- {m}\n{t.to_string()}\n")

table(BASE_PROMPT)

--- CLIP gốc
dataset  RAFDB AffectNet7 AffectNet8 FERPlus  Mean4
config                                             
P0       51.90      40.77      36.45   39.00  42.03
P0+flip  51.90      41.29      36.80   38.69  42.17
P1       47.62      41.11      37.85   45.72  43.08
P1+flip  48.47      41.63      38.25   47.44  43.95
P2       50.06      41.11      37.85   46.17  43.80
P2+flip  51.75      41.63      38.25   47.42  44.76

--- Paper
dataset  RAFDB AffectNet7 AffectNet8 FERPlus  Mean4
config                                             
P0       57.98      44.20      38.92   43.13  46.06
P0+flip  59.22      44.57      39.40   43.29  46.62
P1       56.31      43.40      39.40   47.17  46.57
P1+flip  57.38      43.43      39.43   48.50  47.18
P2       57.17      43.40      39.40   49.38  47.34
P2+flip  58.87      43.43      39.43   49.76  47.87

--- EXP-000
dataset         RAFDB    AffectNet7    AffectNet8       FERPlus         Mean4
config                                               

# Bảng 2 — Δ UAR so với P0 (cùng checkpoint, Ensemble Five)
Tiêu chí cho **P0+flip**: Mean4 ≥ +0.5 trên EXP-000 và EXP-004, không dataset nào < −0.5.

In [9]:
table(BASE_PROMPT, delta=True)

x = u.xs(BASE_PROMPT, level="prompt")
d = (x.xs(MAIN_CFG, level="config") - x.xs(BASE_CFG, level="config")).groupby(level="model").mean()
for m in ["EXP-000", "EXP-004"]:
    ok = d.loc[m, "Mean4"] >= 0.5 and (d.loc[m, DATASETS] >= -0.5).all()
    print(f"{m}: Δ Mean4 = {d.loc[m, 'Mean4']:+.2f}, min Δ dataset = {d.loc[m, DATASETS].min():+.2f} → {'ĐẠT' if ok else 'KHÔNG đạt'}")

--- CLIP gốc
dataset  RAFDB AffectNet7 AffectNet8 FERPlus Mean4
config                                            
P0+flip   0.00       0.51       0.35   -0.31  0.14
P1       -4.28       0.34       1.40    6.72  1.05
P1+flip  -3.43       0.86       1.80    8.44  1.92
P2       -1.85       0.34       1.40    7.18  1.77
P2+flip  -0.16       0.86       1.80    8.42  2.73

--- Paper
dataset  RAFDB AffectNet7 AffectNet8 FERPlus Mean4
config                                            
P0+flip   1.24       0.37       0.48    0.16  0.56
P1       -1.66      -0.80       0.48    4.04  0.51
P1+flip  -0.59      -0.77       0.50    5.37  1.13
P2       -0.80      -0.80       0.48    6.24  1.28
P2+flip   0.90      -0.77       0.50    6.63  1.81

--- EXP-000
dataset         RAFDB    AffectNet7   AffectNet8       FERPlus        Mean4
config                                                                     
P0+flip   1.26 ± 0.54   0.47 ± 0.29  0.22 ± 0.20  -0.10 ± 0.89  0.46 ± 0.43
P1       -3.74 ± 1.57

# Bảng 3 — Ensemble Ten (prompt phụ): UAR và Δ so với P0

In [10]:
table("Expression Ensemble Ten")
table("Expression Ensemble Ten", delta=True)

--- CLIP gốc
dataset  RAFDB AffectNet7 AffectNet8 FERPlus  Mean4
config                                             
P0       53.99      41.69      35.27   38.08  42.26
P0+flip  54.69      41.80      35.42   37.48  42.35
P1       50.40      41.94      37.48   46.14  43.99
P1+flip  50.60      41.80      37.30   47.01  44.18
P2       53.18      41.94      37.48   46.19  44.70
P2+flip  53.47      41.80      37.30   46.30  44.72

--- Paper
dataset  RAFDB AffectNet7 AffectNet8 FERPlus  Mean4
config                                             
P0       58.37      45.00      39.90   43.89  46.79
P0+flip  59.14      45.37      40.28   44.38  47.29
P1       55.92      44.17      39.78   49.66  47.38
P1+flip  56.41      44.60      39.92   49.79  47.68
P2       56.24      44.17      39.78   50.27  47.61
P2+flip  58.01      44.60      39.92   50.26  48.20

--- EXP-000
dataset         RAFDB    AffectNet7    AffectNet8       FERPlus         Mean4
config                                               

# Bảng 4 — Accuracy từng lớp: Δ so với P0 (Ensemble Five, mean seed)
Flip có làm hỏng Contempt (AU một bên) không? Chuẩn hoá đổi lớp nào?

In [11]:
for ds_name in ["RAFDB", "FERPlus"]:
    cols = [f"acc_{c}" for c in classes[ds_name]]
    for m in ["CLIP gốc", "EXP-000", "EXP-004"]:
        r = res[(res.model == m) & (res.dataset == ds_name) & (res.prompt == BASE_PROMPT)]
        t = r.groupby("config")[cols].mean().loc[CFG_NAMES]
        t.columns = classes[ds_name]
        t = pd.concat([t.loc[[BASE_CFG]], (t - t.loc[BASE_CFG]).drop(BASE_CFG).add_prefix("Δ ", axis=0)])
        print(f"--- {m} / {ds_name}\n{t.round(1).to_string()}\n")

--- CLIP gốc / RAFDB
           Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear
config                                                                
P0              17.4     52.1     41.2   90.7      50.8     70.6  40.5
Δ P0+flip       -1.1      1.7     -3.5    0.0      -1.8     -0.6   5.4
Δ P1             6.1    -13.2    -33.2    5.6       3.3     -1.2   2.7
Δ P1+flip        7.4    -10.0    -35.4    4.9       1.8      1.9   5.4
Δ P2             9.0     -6.7    -32.8    5.6       3.3      1.9   6.8
Δ P2+flip       11.1     -2.3    -35.3    6.2       6.1      5.0   8.1

--- EXP-000 / RAFDB
           Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear
config                                                                
P0              87.3     45.7     45.2   88.9      60.2     50.0  29.3
Δ P0+flip        0.2      3.4     -0.8    1.4       0.3     -0.6   5.0
Δ P1             0.8    -13.2    -33.8    3.5       1.7      6.2   8.6
Δ P1+flip        1.2     -7.8    -3

# Lưu JSON (cùng định dạng `research_log/results/`)

In [12]:
out = {"source": "notebooks/EXP_008_test_preprocessing.ipynb",
       "rows": [{"model": r.model, "seed": int(r.seed), "config": r.config, "prompt": r.prompt, "dataset": r.dataset,
                 "UAR": round(r.UAR, 2), "WAR": round(r.WAR, 2)} for r in res.itertuples()]}
with open(f"{OUT_DIR}/EXP-008-test-preprocessing.json", "w") as f:
    json.dump(out, f, indent=1, ensure_ascii=False)
print(f"{len(out['rows'])} dòng → {OUT_DIR}/EXP-008-test-preprocessing.json")

384 dòng → /content/drive/MyDrive/ExpCLIP_output/exp008/EXP-008-test-preprocessing.json
